# Cloud-edge experiment frontend
The sole editable configuration, including operational controls, is `config/experiment.json`. All controls default false. Run All only loads inputs and reads local status when an experiment already exists. Source/config changes invalidate previous approvals; historical artifacts are never rewritten. Resume is a separate explicit operation, never an automatic retry.

## 1. Load and optionally prepare
Preparation creates the frozen plan only when `workflow.experiment.prepare_enabled` is explicitly enabled. Otherwise open an existing local experiment, or leave it unset.

In [ ]:
from pathlib import Path
from cloud_edge_rl.config import load_config
from cloud_edge_rl.experiment import Experiment

PROJECT_ROOT = Path.cwd()
CONFIG = load_config(root=PROJECT_ROOT)
controls = CONFIG['workflow']['experiment']
experiment_directory = PROJECT_ROOT / controls['directory']
experiment = None
RESULTS = {}
if controls['prepare_enabled']:
    experiment = Experiment.prepare(CONFIG, PROJECT_ROOT, directory=experiment_directory)
elif experiment_directory.is_dir():
    experiment = Experiment.open(experiment_directory, root=PROJECT_ROOT)
print({'config': 'config/experiment.json', 'directory': str(experiment_directory), 'experiment_available': experiment is not None})

## 2. Submit
Explicit JSON opt-in only; existing historical approvals do not authorize new execution. Submission remains subject to scoped admission.

In [ ]:
if controls['submission_enabled']:
    if experiment is None:
        raise RuntimeError('Prepare or open an experiment before explicit submission')
    RESULTS['submission'] = experiment.submit()
else:
    print('Submission disabled in config/experiment.json')

## 3. Status
Local status by default (`refresh_enabled: false`); network refresh requires explicit JSON opt-in.

In [ ]:
if experiment is not None:
    RESULTS['status'] = experiment.status(refresh=controls['refresh_enabled'])
else:
    RESULTS['status'] = {'state': 'not_prepared', 'directory': str(experiment_directory)}
print(RESULTS['status'])

## 4. Collect
Artifact collection is disabled unless explicitly enabled in JSON; it is not a retry or a new experiment.

In [ ]:
if controls['collection_enabled']:
    if experiment is None:
        raise RuntimeError('Prepare or open an experiment before explicit collection')
    RESULTS['collection'] = experiment.collect()
else:
    print('Collection disabled in config/experiment.json')

## 5. Verify
Verification is disabled unless explicitly enabled in JSON; reports must distinguish incomplete and failed evidence.

In [ ]:
if controls['verification_enabled']:
    if experiment is None:
        raise RuntimeError('Prepare or open an experiment before explicit verification')
    RESULTS['verification'] = experiment.verify()
else:
    print('Verification disabled in config/experiment.json')

## 6. Results
Display returned status and artifacts only; this cell performs no inference, submission, or implicit resume.

In [ ]:
print(RESULTS)